# <span style=color:crimson>**Part 1:** Imports</span>

In [1]:
#pip install geopandas geodatasets

In [2]:
#Packages:
import pandas as pd
import numpy as np
import plotly

#Geo Packages:
import geopandas as gpd

In [3]:
#files: 
games_df = pd.read_csv("data/old/nintendo_handheld_games.csv")
console_time_df =  pd.read_csv('data/old/console_tempotal.csv')
console_total_df = pd.read_csv('data/old/console_total.csv')
sw_time_df = pd.read_csv('data/old/software_temporal.csv')
sw_total_df = pd.read_csv('data/old/software_total.csv')

---

# <span style=color:crimson>**Part 2:** Browsing Game Data: </span>

In [4]:
games_df.head()

,Pos,Game,Game.1,Console,Publisher,VGChartz Score,Critic Score,User Score,Total Shipped,Release Date,Last Update,Unnamed: 11_level_2,Console_Scraped
0,1,Tetris,Tetris,NaN,Nintendo,NaN,NaN,NaN,35.00m,31st Jul 89,NaN,NaN,GB
1,2,Pokémon Red / Green / Blue Version,Pokémon Red / Green / Blue Version,NaN,Nintendo,NaN,9.4,NaN,31.38m,30th Sep 98,NaN,NaN,GB
2,3,Pokémon Gold / Silver Version,Pokémon Gold / Silver Version,NaN,Nintendo,NaN,9.2,NaN,23.10m,14th Oct 00,NaN,NaN,GB
3,4,Super Mario Land,Super Mario Land,NaN,Nintendo,NaN,NaN,NaN,18.14m,01st Aug 89,NaN,NaN,GB
4,5,Pokémon Yellow: Special Pikachu Edition,Pokémon Yellow: Special Pikachu Edition,NaN,Nintendo,NaN,8.7,NaN,14.64m,19th Oct 99,NaN,NaN,GB


In [5]:
games_df.columns.tolist()

['Pos',
 'Game',
 'Game.1',
 'Console',
 'Publisher',
 'VGChartz Score',
 'Critic Score',
 'User Score',
 'Total Shipped',
 'Release Date',
 'Last Update',
 'Unnamed: 11_level_2',
 'Console_Scraped']

In [6]:
games_df.dtypes

Pos                      int64
Game                    object
Game.1                  object
Console                float64
Publisher               object
VGChartz Score         float64
Critic Score           float64
User Score             float64
Total Shipped           object
Release Date            object
Last Update             object
Unnamed: 11_level_2    float64
Console_Scraped         object
dtype: object

In [7]:
games_df['Console_Scraped'].value_counts()

Console_Scraped
GB     250
GBA    250
DS     250
3DS    250
NS     250
Name: count, dtype: int64

---

# <span style=color:crimson>**Part 3:** Cleaning Game Data</span>

### <span style=color:red>Deleting Columns</span>

In [8]:
#Deleting columns:
games_df = games_df.drop(columns = ['Game.1', 'Console', 'Last Update', 'Unnamed: 11_level_2'])

### <span style=color:red>Renaming Columns</span>

In [9]:
#Renaming and Reordering:
games_df = games_df.rename(columns = {'Pos': 'rank', 'Console_Scraped' : 'console', 'Game' : 'game', 'Publisher' : 'publisher', 
                           'VGChartz Score' : 'vgc_score', 'Critic Score' : 'critic_score', 'User Score' : 'user_score',
                            'Total Shipped' : 'units_sold', 'Release Date': 'release_date' })

### <span style=color:red>Reordering Columns</span>

In [10]:
#Setting an order:
col_order = ['console', 'game', 'rank', 'publisher', 'vgc_score', 'critic_score', 'user_score', 'units_sold', 'release_date']
games_df = games_df[col_order]

In [11]:
games_df.head()

,console,game,rank,publisher,vgc_score,critic_score,user_score,units_sold,release_date
0,GB,Tetris,1,Nintendo,NaN,NaN,NaN,35.00m,31st Jul 89
1,GB,Pokémon Red / Green / Blue Version,2,Nintendo,NaN,9.4,NaN,31.38m,30th Sep 98
2,GB,Pokémon Gold / Silver Version,3,Nintendo,NaN,9.2,NaN,23.10m,14th Oct 00
3,GB,Super Mario Land,4,Nintendo,NaN,NaN,NaN,18.14m,01st Aug 89
4,GB,Pokémon Yellow: Special Pikachu Edition,5,Nintendo,NaN,8.7,NaN,14.64m,19th Oct 99


### <span style=color:red>Datetime Conversion</span>

In [12]:
#Converting Release Date to Datetime:
games_df['release_date'] = pd.to_datetime(games_df['release_date'], format = 'mixed')

### <span style=color:red>Units Sold Conversion</span>

In [13]:
#Converting units sold:
games_df['units_sold'] = (
    games_df['units_sold']
    .astype(str)
    .str.replace('m', '', case=False)
    .astype(float) * 1_000_000
)

In [14]:
games_df.head()

,console,game,rank,publisher,vgc_score,critic_score,user_score,units_sold,release_date
0,GB,Tetris,1,Nintendo,NaN,NaN,NaN,35000000.0,1989-07-31
1,GB,Pokémon Red / Green / Blue Version,2,Nintendo,NaN,9.4,NaN,31380000.0,1998-09-30
2,GB,Pokémon Gold / Silver Version,3,Nintendo,NaN,9.2,NaN,23100000.0,2000-10-14
3,GB,Super Mario Land,4,Nintendo,NaN,NaN,NaN,18140000.0,1989-08-01
4,GB,Pokémon Yellow: Special Pikachu Edition,5,Nintendo,NaN,8.7,NaN,14640000.0,1999-10-19


### <span style=color:red>Checking Null:</span>

In [15]:
games_df[games_df['critic_score'] ==  'NaN'].value_counts().sum()

np.int64(0)

In [16]:
games_df.isna().sum()

console            0
game               0
rank               0
publisher          0
vgc_score       1108
critic_score     779
user_score      1210
units_sold       894
release_date       2
dtype: int64

In [17]:
games_df.groupby('console')['critic_score'].apply(lambda x: x.isna().sum())

console
3DS    106
DS     130
GB     236
GBA    159
NS     148
Name: critic_score, dtype: int64

### <span style=color:red>Deleting VGC Score & User Score:</span>

In [18]:
games_df = games_df.drop(columns = ['vgc_score', 'user_score'])

### <span style=color:red>Correcting Date Inaccuracy:</span>

In [19]:
#Updating a incorrect date: 
games_df.loc[games_df['game'] == 'Dragon Quest Monsters: Caravan Heart', 'release_date'] = '29th Mar 03'

### <span style=color:red>Final Dataset:</span>

In [20]:
games_df.head()

,console,game,rank,publisher,critic_score,units_sold,release_date
0,GB,Tetris,1,Nintendo,NaN,35000000.0,1989-07-31
1,GB,Pokémon Red / Green / Blue Version,2,Nintendo,9.4,31380000.0,1998-09-30
2,GB,Pokémon Gold / Silver Version,3,Nintendo,9.2,23100000.0,2000-10-14
3,GB,Super Mario Land,4,Nintendo,NaN,18140000.0,1989-08-01
4,GB,Pokémon Yellow: Special Pikachu Edition,5,Nintendo,8.7,14640000.0,1999-10-19


# <span style=color:crimson>**Part 4:** Cleaning Other Datasets</span>

## <span style=color:red>Checking Data Types:</span>

### <span style=color:tomato>Consoles by Time:</span>

In [21]:
console_time_df.dtypes

Year        int64
Console    object
Region     object
Sales       int64
dtype: object

### <span style=color:tomato>Total Consoles:</span>

In [22]:
console_total_df.dtypes

Console        object
Model          object
Region         object
Total Sales     int64
dtype: object

### <span style=color:tomato>Software by Time:</span>

In [23]:
sw_time_df.dtypes

Year        int64
Console    object
Region     object
Sales       int64
dtype: object

### <span style=color:tomato>Total Software:</span>

In [24]:
sw_total_df.dtypes

Console        object
Region         object
Total Sales     int64
dtype: object

## <span style=color:red>Appending the Boundaries for Japan, US, and Europe:</span>

### <span style=color:tomato>Finding Boundaries:</span>

In [25]:
#Loading a Dataset with Earth's Boundaries for all countries:
url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
world = gpd.read_file(url)


#AFTER 2017:

#Finding the boundary for the countries/continent:
world['region_modern'] = 'Other'
world.loc[world['ADM0_A3'] == 'USA', 'region_modern'] = 'United States'
world.loc[world['ADM0_A3'] == 'JPN', 'region_modern'] = 'Japan'
world.loc[world['CONTINENT'] == 'Europe', 'region_modern'] = 'Europe'

#Merging boundaries:
modern_map = world.dissolve(by='region_modern')[['geometry']].reset_index()
modern_map = modern_map.rename(columns = {'region_modern': 'Region'})


#BEFORE 2017: 
world['region_old'] = 'Other'
world.loc[world['ADM0_A3'] == 'USA', 'region_old'] = 'United States'
world.loc[world['ADM0_A3'] == 'JPN', 'region_old'] = 'Japan'

#Merging boundaries:
old_map = world.dissolve(by='region_old')[['geometry']].reset_index()
old_map = old_map.rename(columns = {'region_old': 'Region'})


### <span style=color:tomato>Appending the Boundaries:</span>

In [26]:
#list of dfs:
all_dfs = [console_time_df, console_total_df, sw_time_df, sw_total_df]

def append_boundaries(dfs, old_map, modern_map, crs):
    updated_dfs = []
    for df in dfs:
        #Splitting DFs based on console:
        old = df[~df['Console'].isin(['Switch', 'Switch 2'])].copy()
        new = df[df['Console'].isin(['Switch', 'Switch 2'])].copy()

        #Merging:
        old_gdf = old.merge(old_map, on ='Region', how = 'left')
        new_gdf = new.merge(modern_map, on ='Region', how = 'left')

        #Combining:
        combined_df = pd.concat([old_gdf, new_gdf], ignore_index = True)
        final_gdf = gpd.GeoDataFrame(combined_df, geometry='geometry', crs = crs)

        updated_dfs.append(final_gdf)
        
    return updated_dfs

In [27]:
console_time_gdf, console_total_gdf, sw_time_gdf, sw_total_gdf = append_boundaries(all_dfs, old_map, modern_map, world.crs)

In [28]:
console_time_gdf.tail(10)

,Year,Console,Region,Sales,geometry
193,2026,Switch,Japan,1140000,"MULTIPOLYGON (((132.92437 34.0603, 133.49297 3..."
194,2026,Switch,United States,1300000,"MULTIPOLYGON (((-154.80741 19.50871, -154.8314..."
195,2026,Switch,Europe,830000,"MULTIPOLYGON (((-53.55484 2.3349, -53.77852 2...."
196,2026,Switch,Other,530000,"MULTIPOLYGON (((-162.43985 -79.28146, -163.027..."
197,2026,Switch,Total,3800000,None
198,2026,Switch 2,Japan,5660000,"MULTIPOLYGON (((132.92437 34.0603, 133.49297 3..."
199,2026,Switch 2,United States,6730000,"MULTIPOLYGON (((-154.80741 19.50871, -154.8314..."
200,2026,Switch 2,Europe,4400000,"MULTIPOLYGON (((-53.55484 2.3349, -53.77852 2...."
201,2026,Switch 2,Other,3060000,"MULTIPOLYGON (((-162.43985 -79.28146, -163.027..."
202,2026,Switch 2,Total,19860000,None


In [29]:
games_df.dtypes

console                 object
game                    object
rank                     int64
publisher               object
critic_score           float64
units_sold             float64
release_date    datetime64[ns]
dtype: object

# <span style=color:crimson>**Part 5:** Exporting All Datasets</span>

In [30]:
# #game_df:
# games_df.to_csv('game_list.csv',index = False)

# #console_by_year:
# console_time_gdf.to_file('console_by_year.geojson',index = False)

# #console_total_sales:
# console_total_gdf.to_file('console_total_sales.geojson',index = False)

# #software_by_year:
# sw_time_gdf.to_file('software_by_year.geojson',index = False)

# #software_total_sales:
# sw_total_gdf.to_file('software_total_sales.geojson', index = False)